In [2]:
import sys, os
import torch
import json
from pathlib import Path
from transformers import AutoTokenizer

# hydra imports; not really required if you will hard-code model params in future
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

# set GENALM_HOME environment variable to point to GENA_LM repo root; required to process config files
GENA_HOME = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM"
os.environ["GENALM_HOME"] = GENA_HOME
sys.path.append(GENA_HOME)

# import model
from downstream_tasks.expression_prediction.expression_model_final import ExpressionCounts
from downstream_tasks.expression_prediction.expression_dataset_final import ExpressionDataset

/home/jovyan/miniconda3/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# we have model parameters and other variables in config files; I made one for inference
GENA_HOME = "/home/jovyan/shares/SR003.nfs2/aspeedok"
os.environ["GENALM_HOME"] = GENA_HOME
sys.path.append(GENA_HOME)

experiment_config = "/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/configs/final_02062026.yaml"

experiment_config_path = Path(experiment_config).expanduser().absolute()

with initialize_config_dir(str(experiment_config_path.parents[0])):
	experiment_config = compose(config_name=experiment_config_path.name)

model_kwargs = instantiate(experiment_config["model_kwargs"])

# initialize model
model = ExpressionCounts(**model_kwargs)

/tmp/ipykernel_3863817/2872936261.py:10: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(experiment_config_path.parents[0])):
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


Using ModernGENA from /home/jovyan/shares/SR003.nfs2/aspeedok/models/moderngena_large
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25.self_attn.q_proj.we

In [4]:
# load checkpoint
CHECKPOINT_PATH = "/home/jovyan/shares/SR003.nfs2/aspeedok/runs/mikhail_experements/super_model_02.06_res2/pytorch_model.bin"
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True))



<All keys matched successfully>

In [5]:
# prepare tokenizers
dna_tokenizer = experiment_config["args_params"]["gen_tokenizer"]
text_tokenizer = experiment_config["shared_dataset_params"]["text_tokenizer"]
dna_tokenizer = AutoTokenizer.from_pretrained(dna_tokenizer)
text_tokenizer = AutoTokenizer.from_pretrained(text_tokenizer, padding_side='left')

dna_max_seq_len = experiment_config["args_params"]["input_seq_len"]
text_max_seq_len = experiment_config["shared_dataset_params"]["text_max_seq_len"]

In [ ]:
0%|          | 0/5000 [00:00<?, ?it/s]-4.8125 0.32272727272727275 {'McKellar2021_Myonuclei_Type_IIb': 10.875, 'McKellar2021_Myonuclei_Type_IIx': 12.6875, 'McKellar2021_Endothelial_Artery': 4.8125, 'McKellar2021_Endothelial_Capillary': 4.78125, 'McKellar2021_Endothelial_Vein': 4.625, 'McKellar2021_M2_Macro._Cx3cr1_hi': 5.96875, 'McKellar2021_M2_Macro._Cx3cr1_lo': 5.40625, 'McKellar2021_Smooth_Muscle_&_Pericytes': 6.0625, 'McKellar2021_MuSCs': 4.1875, 'McKellar2021_FAPs_Adipogenic': 2.71875, 'McKellar2021_FAPs_Pro-remodeling': 2.953125, 'McKellar2021_FAPs_Stem': 3.765625}
 10%|█         | 501/5000 [01:21<09:51,  7.60it/s]-170.0 0.3090909090909091 {'McKellar2021_Myonuclei_Type_IIb': 202.0, 'McKellar2021_Myonuclei_Type_IIx': 215.0, 'McKellar2021_Endothelial_Artery': 5.6875, 'McKellar2021_Endothelial_Capillary': 16.75, 'McKellar2021_Endothelial_Vein': 8.8125, 'McKellar2021_M2_Macro._Cx3cr1_hi': 32.0, 'McKellar2021_M2_Macro._Cx3cr1_lo': 21.625, 'McKellar2021_Smooth_Muscle_&_Pericytes': 18.0, 'McKellar2021_MuSCs': 5.8125, 'McKellar2021_FAPs_Adipogenic': -1.078125, 'McKellar2021_FAPs_Pro-remodeling': 2.15625, 'McKellar2021_FAPs_Stem': 1.6484375}
 20%|██        | 1000/5000 [02:13<05:26, 12.25it/s]-400.0 0.30454545454545456 {'McKellar2021_Myonuclei_Type_IIb': 450.0, 'McKellar2021_Myonuclei_Type_IIx': 478.0, 'McKellar2021_Endothelial_Artery': -4.5625, 'McKellar2021_Endothelial_Capillary': 19.125, 'McKellar2021_Endothelial_Vein': 3.375, 'McKellar2021_M2_Macro._Cx3cr1_hi': 50.0, 'McKellar2021_M2_Macro._Cx3cr1_lo': 27.5, 'McKellar2021_Smooth_Muscle_&_Pericytes': 36.75, 'McKellar2021_MuSCs': 21.5, 'McKellar2021_FAPs_Adipogenic': -27.875, 'McKellar2021_FAPs_Pro-remodeling': -3.0625, 'McKellar2021_FAPs_Stem': 2.4375}
 30%|███       | 1501/5000 [02:56<04:18, 13.52it/s]-520.5 0.3090909090909091 {'McKellar2021_Myonuclei_Type_IIb': 544.0, 'McKellar2021_Myonuclei_Type_IIx': 580.0, 'McKellar2021_Endothelial_Artery': -9.0, 'McKellar2021_Endothelial_Capillary': -1.125, 'McKellar2021_Endothelial_Vein': -6.25, 'McKellar2021_M2_Macro._Cx3cr1_hi': 23.5, 'McKellar2021_M2_Macro._Cx3cr1_lo': 7.9375, 'McKellar2021_Smooth_Muscle_&_Pericytes': 10.625, 'McKellar2021_MuSCs': -9.3125, 'McKellar2021_FAPs_Adipogenic': -18.25, 'McKellar2021_FAPs_Pro-remodeling': -19.375, 'McKellar2021_FAPs_Stem': -10.75}
 40%|████      | 2001/5000 [03:36<04:22, 11.41it/s]-555.375 0.3090909090909091 {'McKellar2021_Myonuclei_Type_IIb': 580.0, 'McKellar2021_Myonuclei_Type_IIx': 600.0, 'McKellar2021_Endothelial_Artery': -8.0625, 'McKellar2021_Endothelial_Capillary': -1.3828125, 'McKellar2021_Endothelial_Vein': -6.71875, 'McKellar2021_M2_Macro._Cx3cr1_hi': 24.625, 'McKellar2021_M2_Macro._Cx3cr1_lo': 6.125, 'McKellar2021_Smooth_Muscle_&_Pericytes': 9.3125, 'McKellar2021_MuSCs': -5.3125, 'McKellar2021_FAPs_Adipogenic': -14.0625, 'McKellar2021_FAPs_Pro-remodeling': -17.75, 'McKellar2021_FAPs_Stem': -12.25}
 50%|█████     | 2501/5000 [04:14<03:36, 11.55it/s]-567.125 0.31363636363636366 {'McKellar2021_Myonuclei_Type_IIb': 580.0, 'McKellar2021_Myonuclei_Type_IIx': 608.0, 'McKellar2021_Endothelial_Artery': -0.0152587890625, 'McKellar2021_Endothelial_Capillary': 7.21875, 'McKellar2021_Endothelial_Vein': 2.1875, 'McKellar2021_M2_Macro._Cx3cr1_hi': 9.0, 'McKellar2021_M2_Macro._Cx3cr1_lo': -4.09375, 'McKellar2021_Smooth_Muscle_&_Pericytes': 12.875, 'McKellar2021_MuSCs': -11.0, 'McKellar2021_FAPs_Adipogenic': -9.625, 'McKellar2021_FAPs_Pro-remodeling': -20.25, 'McKellar2021_FAPs_Stem': -15.8125}
 60%|██████    | 3002/5000 [04:51<02:24, 13.86it/s]-570.75 0.30454545454545456 {'McKellar2021_Myonuclei_Type_IIb': 608.0, 'McKellar2021_Myonuclei_Type_IIx': 612.0, 'McKellar2021_Endothelial_Artery': 0.006378173828125, 'McKellar2021_Endothelial_Capillary': 14.125, 'McKellar2021_Endothelial_Vein': 5.90625, 'McKellar2021_M2_Macro._Cx3cr1_hi': 37.25, 'McKellar2021_M2_Macro._Cx3cr1_lo': 14.1875, 'McKellar2021_Smooth_Muscle_&_Pericytes': 18.125, 'McKellar2021_MuSCs': -0.08642578125, 'McKellar2021_FAPs_Adipogenic': -16.75, 'McKellar2021_FAPs_Pro-remodeling': -12.3125, 'McKellar2021_FAPs_Stem': -2.75}
 66%|██████▌   | 3275/5000 [05:11<02:43, 10.53it/s]END of generation
[ScoredSeq(seq='AACAGAGGTGCTGGGACCGGCTAGGGGAGCTACGACTATATTTCGACGAATTGCTAATCTAAAGCCGAACACCCGCAAGGGACGCCCGCTCTGATCTGACGTCCCGGCCTACATCCCTTCGCAGGCCCCTCCCACCCGCTCCGGCCTGCGCCACGGCACAGAGCTTTTACGCATGCGACCCCGAGCCCCGCGCAAACCCAGTGGCCGGGCGACGTCGCGG', dt={'McKellar2021_Myonuclei_Type_IIb': 608.0, 'McKellar2021_Myonuclei_Type_IIx': 612.0, 'McKellar2021_Endothelial_Artery': 0.006378173828125, 'McKellar2021_Endothelial_Capillary': 14.125, 'McKellar2021_Endothelial_Vein': 5.90625, 'McKellar2021_M2_Macro._Cx3cr1_hi': 37.25, 'McKellar2021_M2_Macro._Cx3cr1_lo': 14.1875, 'McKellar2021_Smooth_Muscle_&_Pericytes': 18.125, 'McKellar2021_MuSCs': -0.08642578125, 'McKellar2021_FAPs_Adipogenic': -16.75, 'McKellar2021_FAPs_Pro-remodeling': -12.3125, 'McKellar2021_FAPs_Stem': -2.75}, score=-570.75, method='snp')]

In [6]:
# Score generated sequence: muscle_miofibers vs other muscle cell groups

generated_sequence = "AACAGAGGTGCTGGGACCGGCTAGGGGAGCTACGACTATATTTCGACGAATTGCTAATCTAAAGCCGAACACCCGCAAGGGACGCCCGCTCTGATCTGACGTCCCGGCCTACATCCCTTCGCAGGCCCCTCCCACCCGCTCCGGCCTGCGCCACGGCACAGAGCTTTTACGCATGCGACCCCGAGCCCCGCGCAAACCCAGTGGCCGGGCGACGTCGCGG"

TARGETS = [
    "McKellar2021_Myonuclei_Type_IIb",
    "McKellar2021_Myonuclei_Type_IIx",
]

OFF_TARGETS = [
    "McKellar2021_Endothelial_Artery",
    "McKellar2021_Endothelial_Capillary",
    "McKellar2021_Endothelial_Vein",
    "McKellar2021_M2_Macro._Cx3cr1_hi",
    "McKellar2021_M2_Macro._Cx3cr1_lo",
    "McKellar2021_Smooth_Muscle_&_Pericytes",
    "McKellar2021_MuSCs",
    "McKellar2021_FAPs_Adipogenic",
    "McKellar2021_FAPs_Pro-remodeling",
    "McKellar2021_FAPs_Stem",
]

selected_keys = TARGETS + OFF_TARGETS
metadata_dir = Path("/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/datasets/data/scRNA_McKellar2021_Smooth_Muscle/metadata")

# Build descriptions exactly through the dataset helper.
description_texts = []
for key in selected_keys:
    metadata_path = metadata_dir / f"{key}.json"
    with open(metadata_path, "r", encoding="utf-8") as handle:
        meta = json.load(handle)
    description_texts.append(
        ExpressionDataset.make_description_from_json(
            meta=meta,
            description_id=key,
            meta_path=str(metadata_path),
        )
    )

# One DNA sequence scored against N descriptions: B=1, N=len(selected_keys).
dna_encoding = dna_tokenizer(
    generated_sequence,
    truncation=True,
    padding="max_length",
    max_length=dna_max_seq_len,
    return_tensors="pt",
)

desc_encoding = text_tokenizer(
    description_texts,
    truncation=True,
    padding="max_length",
    max_length=text_max_seq_len,
    return_tensors="pt",
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.eval().to(device)

n_keys = len(selected_keys)
input_ids = dna_encoding["input_ids"].repeat(n_keys, 1).to(device)
attention_mask = dna_encoding["attention_mask"].repeat(n_keys, 1).to(device)
desc_input_ids = desc_encoding["input_ids"].unsqueeze(0).to(device)
desc_attention_mask = desc_encoding["attention_mask"].unsqueeze(0).to(device)
dataset_flag = torch.ones((1, n_keys), device=device, dtype=torch.bool)

with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        desc_input_ids=desc_input_ids,
        desc_attention_mask=desc_attention_mask,
        dataset_flag=dataset_flag,
    )

preds = output["logits"][:, 0, 0].detach().float().cpu()
dt = {key: preds[i].item() for i, key in enumerate(selected_keys)}

target_value = min(dt[key] for key in TARGETS)
off_target_value = max(dt[key] for key in OFF_TARGETS)
score = -(target_value - off_target_value)

print("sequence length:", len(generated_sequence))
print("target min:", target_value)
print("off-target max:", off_target_value)
print("score:", score)

import pandas as pd
score_df = pd.DataFrame([
    {
        "track": key,
        "group": "target" if key in TARGETS else "off_target",
        "prediction": dt[key],
    }
    for key in selected_keys
])
score_df



sequence length: 220
target min: 0.68359375
off-target max: 0.6953125
score: 0.01171875


,track,group,prediction
0,McKellar2021_Myonuclei_Type_IIb,target,0.683594
1,McKellar2021_Myonuclei_Type_IIx,target,0.699219
2,McKellar2021_Endothelial_Artery,off_target,0.578125
3,McKellar2021_Endothelial_Capillary,off_target,0.511719
4,McKellar2021_Endothelial_Vein,off_target,0.585938
5,McKellar2021_M2_Macro._Cx3cr1_hi,off_target,0.667969
6,McKellar2021_M2_Macro._Cx3cr1_lo,off_target,0.695312
7,McKellar2021_Smooth_Muscle_&_Pericytes,off_target,0.628906
8,McKellar2021_MuSCs,off_target,0.554688
9,McKellar2021_FAPs_Adipogenic,off_target,0.613281


In [7]:
# Diagnostics: inspect exact model inputs used by this notebook scoring cell
import json
from pathlib import Path

DEBUG_SEQUENCE = "GCCGTGTGCAGGCTAGCCGGCGTGTCGCTCTGATGCTCGGCGAGATGACTGGCGGGGAGTGCGCACGGCTCATCGCCCCTGCGGTCGCGACTGGCCAGTTCTGCTGGAGGCGCGCCTCCCCAGAGACTGAGGTAAGTACTTCCCTGTGCGCCGCGGAAGAGCGAGTGAGAGATGGGCGGGGCAGTCGCCAGATGACGTCAGTCTCGCGCGCGGGCGGAGG"
print("=== INFERENCE_MINJA INPUT DIAGNOSTICS ===")
print("experiment_config:", globals().get("experiment_config", "<missing>"))
print("checkpoint:", globals().get("CHECKPOINT_PATH", globals().get("checkpoint_path", "<missing>")))
print("device:", globals().get("device", "<not set yet>"))
print("dna tokenizer:", getattr(dna_tokenizer, "name_or_path", type(dna_tokenizer).__name__))
print("text tokenizer:", getattr(text_tokenizer, "name_or_path", type(text_tokenizer).__name__))
print("dna_max_seq_len:", dna_max_seq_len)
print("text_max_seq_len:", text_max_seq_len)
print("selected_keys:", selected_keys if "selected_keys" in globals() else "<missing; run scoring cell first or this cell defines below>")

TARGETS_DIAG = [
    "McKellar2021_Myonuclei_Type_IIb",
    "McKellar2021_Myonuclei_Type_IIx",
]
OFF_TARGETS_DIAG = [
    "McKellar2021_Endothelial_Artery",
    "McKellar2021_Endothelial_Capillary",
    "McKellar2021_Endothelial_Vein",
    "McKellar2021_M2_Macro._Cx3cr1_hi",
    "McKellar2021_M2_Macro._Cx3cr1_lo",
    "McKellar2021_Smooth_Muscle_&_Pericytes",
    "McKellar2021_MuSCs",
    "McKellar2021_FAPs_Adipogenic",
    "McKellar2021_FAPs_Pro-remodeling",
    "McKellar2021_FAPs_Stem",
]
selected_keys_diag = TARGETS_DIAG + OFF_TARGETS_DIAG
metadata_dir = Path("/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/datasets/data/scRNA_McKellar2021_Smooth_Muscle/metadata")
print("diagnostic selected_keys:", selected_keys_diag)
print("metadata_dir:", metadata_dir)
print("sequence length:", len(DEBUG_SEQUENCE))

# Current inference_minja scoring behavior.
dna_encoded_current = dna_tokenizer(
    DEBUG_SEQUENCE,
    truncation=True,
    padding="max_length",
    max_length=dna_max_seq_len,
    return_tensors="pt",
)
print("--- current inference_minja DNA tokenization ---")
print("DNA add_special_tokens:", True)
print("DNA padding:", "max_length")
print("DNA input_ids shape before repeat:", tuple(dna_encoded_current["input_ids"].shape))
print("DNA attention_mask sum before repeat:", int(dna_encoded_current["attention_mask"].sum()))
print("DNA first 20 ids:", dna_encoded_current["input_ids"][0, :20].tolist())
print("DNA last 20 ids:", dna_encoded_current["input_ids"][0, -20:].tolist())

# main_gen/genalg behavior for direct comparison.
dna_encoded_like_main = dna_tokenizer(
    DEBUG_SEQUENCE,
    add_special_tokens=False,
    return_attention_mask=True,
    return_tensors="pt",
)
print("--- main_gen-like DNA tokenization ---")
print("DNA add_special_tokens:", False)
print("DNA padding:", None)
print("DNA input_ids shape before repeat:", tuple(dna_encoded_like_main["input_ids"].shape))
print("DNA attention_mask sum before repeat:", int(dna_encoded_like_main["attention_mask"].sum()))
print("DNA first 20 ids:", dna_encoded_like_main["input_ids"][0, :20].tolist())
print("DNA last 20 ids:", dna_encoded_like_main["input_ids"][0, -20:].tolist())

description_texts = []
for key in selected_keys_diag:
    metadata_path = metadata_dir / f"{key}.json"
    with open(metadata_path, "r", encoding="utf-8") as handle:
        meta = json.load(handle)
    description_texts.append(
        ExpressionDataset.make_description_from_json(
            meta=meta,
            description_id=key,
            meta_path=str(metadata_path),
        )
    )

desc_encoded_current = text_tokenizer(
    description_texts,
    truncation=True,
    padding="max_length",
    max_length=text_max_seq_len,
    return_tensors="pt",
)
desc_encoded_like_main = text_tokenizer(
    description_texts,
    padding=True,
    truncation=True,
    max_length=text_max_seq_len,
    return_tensors="pt",
)
print("--- descriptions ---")
print("current description padding:", "max_length")
print("current desc shape:", tuple(desc_encoded_current["input_ids"].shape))
print("current desc attention sums:", desc_encoded_current["attention_mask"].sum(dim=1).tolist())
print("main-like description padding:", True)
print("main-like desc shape:", tuple(desc_encoded_like_main["input_ids"].shape))
print("main-like desc attention sums:", desc_encoded_like_main["attention_mask"].sum(dim=1).tolist())
print("first description prefix:", description_texts[0][:300])
print("current first desc ids head/tail:", desc_encoded_current["input_ids"][0, :20].tolist(), desc_encoded_current["input_ids"][0, -20:].tolist())
print("main-like first desc ids head/tail:", desc_encoded_like_main["input_ids"][0, :20].tolist(), desc_encoded_like_main["input_ids"][0, -20:].tolist())

def run_debug_forward(dna_encoded, desc_encoded, label):
    device_local = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device_local)
    n_keys = len(selected_keys_diag)
    model_inputs = {
        "input_ids": dna_encoded["input_ids"].repeat(n_keys, 1).to(device_local),
        "attention_mask": dna_encoded["attention_mask"].repeat(n_keys, 1).to(device_local),
        "desc_input_ids": desc_encoded["input_ids"].unsqueeze(0).to(device_local),
        "desc_attention_mask": desc_encoded["attention_mask"].unsqueeze(0).to(device_local),
        "dataset_flag": torch.ones((1, n_keys), dtype=torch.bool, device=device_local),
    }
    print(f"--- forward {label} ---")
    for name, tensor in model_inputs.items():
        print(name, tuple(tensor.shape), tensor.dtype, tensor.device)
    print("dataset_flag:", model_inputs["dataset_flag"].detach().cpu().int().tolist())
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
        out = model(**model_inputs)
    preds = out["logits"][:, 0, 0].detach().float().cpu()
    dt = {key: preds[i].item() for i, key in enumerate(selected_keys_diag)}
    target_value = min(dt[key] for key in TARGETS_DIAG)
    off_target_value = max(dt[key] for key in OFF_TARGETS_DIAG)
    score = -(target_value - off_target_value)
    print("predictions:", dt)
    print("target min:", target_value)
    print("off-target max:", off_target_value)
    print("score:", score)
    return dt, score

current_dt, current_score = run_debug_forward(dna_encoded_current, desc_encoded_current, "current inference_minja")
main_like_dt, main_like_score = run_debug_forward(dna_encoded_like_main, desc_encoded_like_main, "main_gen-like")
print("score difference current - main_like:", current_score - main_like_score)



=== INFERENCE_MINJA INPUT DIAGNOSTICS ===
experiment_config: {'TASK_NAME': 'expression/final_02062026', 'HOME_PATH': '${oc.env:GENALM_HOME}', 'MODEL_PATH_ROOT': '${HOME_PATH}/runs/${TASK_NAME}', 'args_params': {'_target_': 'builtins.dict', 'resume': '/20260602-193222/model_107500/pytorch_model.bin', 'model_cfg': './data/configs/L12-H768-A12-V32k-preln.json', 'model_cls': 'downstream_tasks.expression_prediction.expression_model_final:ExpressionCounts', 'model_path': '${MODEL_PATH_ROOT}', 'clip_grad_norm': 1, 'input_seq_len': 1024, 'optimizer': 'AdamW', 'weight_decay': 0.0001, 'seed': 45, 'num_warmup_steps': 10000, 'log_interval': 500, 'valid_interval': 500, 'save_best': True, 'save_interval': 2500, 'optimize_metric': 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset', 'optimize_mode': 'max', 'data_n_workers': 8, 'lr': 0.0001, 'lr_scheduler': 'constant_with_warmup', 'reset_lr': True, 'reset_optimizer': True, 'reset_iteration': True, 'early_stopping_patience': 10000000, 'iters':